# Chapter 6 — RAG: Grounding Aegis in Your Runbooks

Aegis knows how to investigate. It does not know **your** incident-response runbooks,
your CVE advisories, or the policy your company adopted last quarter. No model does,
and no amount of prompting will fix that — the knowledge has to be retrieved.

Retrieval-augmented generation is the standard answer: index your documents, retrieve
the relevant passages when a question arrives, and require the agent to answer from
what was retrieved rather than from what it happens to believe.

The pipeline has four stages, and almost everyone focuses on the wrong one:

```
chunk  ->  embed  ->  retrieve  ->  ground
```

Teams argue about embedding models. The stage that decides whether retrieval finds
the right passage is the *first* one — **chunking**.

This lab is a guidebook in two halves.

**Part 1 — RAG from first principles.** The corpus, four chunking strategies, an
embedder, an index, grounding with citations, and a comparison that looks like a win
and is actually a trap.

**Part 2 — Advanced RAG.** Query transformation, hybrid dense + sparse retrieval,
reranking, parent-document retrieval, a real vector database with metadata filters,
and the retrieval metrics that turn "it seems to work" into a number.

Everything runs offline and deterministically on the `mock` tier. Where a real system
would call a model — to embed, to rewrite a query, to rerank, to generate — the lab
uses a deterministic stand-in with the *same interface*, and says so each time.


## Setup

Every lab in this book installs from **one** `requirements.txt` in the companion
repository. No notebook pins its own versions: change a dependency there and it
changes everywhere, including CI.

The clone below fails loudly on purpose. A setup step that swallows its own error
surfaces later as a confusing `ModuleNotFoundError`, and you waste an hour looking in
the wrong place.


In [ ]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


In [ ]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt


Now verify the environment before running any lab code. This is the same check CI
runs. It catches the one dependency conflict that would otherwise waste your
afternoon, and it confirms this chapter's source folder is in the checkout.


In [ ]:
!python tools/check_env.py --chapter 6


### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to `openai`,
the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the key
icon in the sidebar (Secrets); the cell below reads it without printing it.


In [ ]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


---
# Part 1 — RAG from first principles

## The corpus

Three incident-response runbooks and two CVE advisories. Small enough to read in
full, which is deliberate — you should be able to check the retriever's work by eye.

Note the structure: every runbook is a numbered sequence of steps. That structure is
going to matter enormously, and generic chunkers are blind to it.


In [ ]:
import sys
sys.path.insert(0, "labs/chapter-06-rag-grounding-aegis")   # this chapter's source lives beside the notebook

from data.corpus import RUNBOOKS, ADVISORIES, METADATA, GOLDEN_SET, all_docs

docs = all_docs()
for doc_id, text in docs.items():
    print(f"{doc_id:22} {len(text):4} chars   {METADATA[doc_id]['type']:9} severity={METADATA[doc_id]['severity']}")

print()
print(RUNBOOKS["rb_account_takeover"])


## Stage 1 — Chunking

You cannot retrieve a whole document; you retrieve a passage. Chunking is how you
decide what a passage *is*, and there is no neutral choice.

Four strategies, each trading precision against context:

1. **Fixed** — cut every N characters. Simple, and blind: it will split a sentence, a
   word, or a runbook step straight down the middle.
2. **Sentence-window** — one chunk per sentence, plus its neighbours. The target
   sentence stays precise for matching; the window restores the context needed to
   answer from it.
3. **Semantic** — group consecutive sentences so a coherent unit stays together.
4. **Hierarchical** — index small children, return the parent section. Sharp
   matching, full-context generation.


In [ ]:
from rag.pipeline import (STRATEGIES, chunk_fixed, chunk_sentence_window,
                          chunk_semantic, chunk_hierarchical)

doc = RUNBOOKS["rb_account_takeover"]

print(f'{"strategy":18} {"chunks":>7} {"avg chars":>10}')
for name, chunker in STRATEGIES.items():
    chunks = chunker(doc)
    avg = sum(len(c) for c in chunks) / len(chunks)
    print(f'{name:18} {len(chunks):7} {avg:10.0f}')


### Look at what fixed chunking actually does

The numbers above are abstract. Print the chunks and the problem becomes obvious.


In [ ]:
print("FIXED (120 chars, blind to meaning):")
for chunk in chunk_fixed(doc)[:3]:
    print(f"  |{chunk}|")

print()
print("SENTENCE-WINDOW (one step, plus its neighbours):")
for chunk in chunk_sentence_window(doc)[:2]:
    print(f"  |{chunk}|")


Look at where the cuts fall. The first chunk ends with an orphaned **"Step 2:"** — a
label with no instruction attached. The third chunk *starts mid-word* ("ication logs"),
and Step 3 is now split across two passages, neither of which is complete.

In an incident that is not an academic problem. It is the agent telling a responder to
"review authentication logs for the" and stopping, because the rest of the sentence was
in a chunk that did not get retrieved. The overlap parameter papers over some of this
and none of it reliably.


## Stage 2 — Embedding

An embedding turns text into a vector so that similar text lands nearby. Production
systems use a trained embedding model; here we use term-frequency vectors and cosine
similarity.

That substitution is deliberate and worth being honest about. It is weaker than a real
embedder — it matches on shared words, not on meaning, so it will miss paraphrase.
What it *does* do is run offline, deterministically, in this notebook, and it
demonstrates the chunking effect faithfully. The interface is identical, so swapping
in a real embedder is a body swap, not a rewrite.


In [ ]:
from rag.pipeline import tokens, embed, cosine

a = embed("disable the account and revoke active sessions")
b = embed("revoke sessions and disable the affected account")
c = embed("upgrade AcmeVPN and rotate session tokens")

print(f"same meaning, different words : {cosine(a, b):.3f}")
print(f"different topic               : {cosine(a, c):.3f}")
print()
print("what the embedder sees:", dict(embed("disable the account and revoke active sessions")))


## Stage 3 — The index and the retriever

Two phases. **Indexing** happens once: chunk every document, embed every chunk, keep
them. **Retrieval** happens per query: embed the question, compare, return the
closest.

That is the whole of a vector database, conceptually. The commercial ones add
persistence, scale, filtering, and speed — not a different idea. (Part 2 uses a real
one.)


In [ ]:
from rag.pipeline import VectorIndex

index = VectorIndex().index(docs, strategy="semantic", metadata=METADATA)
print(f"{len(index.chunks)} chunks indexed across {len(docs)} documents")

query = "how do I contain an account takeover"
for score, chunk in index.retrieve(query, k=3):
    print(f"  [{score:.3f}] {chunk.doc_id:22} {chunk.text[:70]}...")


## Stage 4 — Grounding

This is the step that makes it RAG rather than search. The retrieved passages are not
a hint or a suggestion — they *are* the material the answer must be built from.

An agent that retrieves context and then answers from its training anyway has not
been grounded. It has been given a reading list it ignored.

Two things below that a search engine does not do:

- **Citations.** The answer names the documents it drew from, so a responder can check.
- **A grounding check.** Every sentence of the answer is tested against the retrieved
  context. A sentence the context does not support is flagged, not shipped.

The "model" here is a deterministic extractive stand-in. Watch what happens when it is
told to do what real models do under pressure — add something plausible that was
never retrieved.


In [ ]:
from rag.advanced import grounded_answer, grounding_check

QUERY = "how do I contain an account takeover and check for data egress"
hits = index.retrieve(QUERY, k=2)

print("query:", QUERY)
print()
for score, chunk in hits:
    print(f"  [{score:.3f}] {chunk.doc_id}")
    print(f"          {chunk.text[:96]}...")

answer = grounded_answer(QUERY, hits)
print()
print("grounded answer:")
print(" ", answer.text)
print("  cites:", answer.citations)
print("  every sentence supported by the context:", answer.supported)

drifted = grounded_answer(QUERY, hits, fabricate=True)
print()
print("the same answer after the model 'helpfully' adds a sentence:")
print("  supported:", drifted.supported)
print("  flagged:  ", drifted.unsupported)


That flagged sentence is the entire risk of RAG in one line. It is fluent, it is
plausible, it is about security — and nothing in the corpus says it. Without the
check it goes into an incident ticket as if a runbook had recommended it.

Chapter 10 measures how often a real model does this. Here the point is that the check
is cheap, deterministic, and belongs *inside* the pipeline, not in a post-mortem.


## The comparison

Now the chapter's actual question. Index the same corpus four times, once per
strategy, and ask a precise, fact-level question. Which strategy retrieves the right
runbook, and how *confidently*?


In [ ]:
from rag.pipeline import evaluate_strategies

PRECISE_QUERY = "detonate URLs in a sandbox and do not visit them directly"
EXPECTED = "rb_phishing"

results = evaluate_strategies(docs, PRECISE_QUERY, EXPECTED)

print(f"query: {PRECISE_QUERY!r}")
print(f"expected document: {EXPECTED}")
print()
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["score"]):
    mark = "correct" if r["correct"] else "WRONG"
    print(f"  {name:16} -> {r['top_doc']:22} score={r['score']:.3f}  {mark}")

spread = max(r["score"] for r in results.values()) - min(r["score"] for r in results.values())
print()
print(f"every strategy is correct. the confidence spread is {spread:.3f}.")


Every strategy retrieved the right runbook. **And that result is a trap.**

### Why this measurement will mislead you

On a five-document corpus, every reasonable chunking strategy retrieves the right
document, because there are only five things it could possibly return. The
hit-or-miss column tells you nothing.

Look at the **confidence spread** instead. That gap is the signal, and here is why.

Retrieval does not return "the right document." It returns whatever scored highest.
On five documents, the weakest strategy still wins comfortably. On forty thousand
documents, hundreds of chunks will score above it by coincidence — and the correct one
is now somewhere in the noise.

The margin *is* the safety buffer. A strategy that wins by a nose today loses at scale,
and it fails silently: no error, no exception, just a confidently wrong answer
grounded in the wrong runbook.

This is the most expensive lesson in the chapter, and you can only see it by looking
at a number that a pass/fail test would have thrown away.


## The strategy nobody writes

Every strategy above is generic — none of them know that a runbook has *steps*.

But our documents do. Splitting on the document's own structure keeps each step whole,
which is exactly what an agent needs when it is telling a human what to do next in an
incident.


In [ ]:
from rag.pipeline import chunk_by_step

chunks = chunk_by_step(RUNBOOKS["rb_account_takeover"])

print(f"{len(chunks)} chunks, one per step (plus the title):")
for chunk in chunks:
    print(f"  |{chunk[:72]}|")

step_counts = [c.count("Step") for c in chunks if "Step" in c]
print()
print("steps per chunk:", step_counts, "(all 1 = no step was split or merged)")


Compare that to the fixed chunker's output earlier, which cut Step 2 in half.

Structure-aware chunking usually beats every generic strategy on documents that have
structure — and most enterprise documents do: runbooks with steps, policies with
clauses, advisories with sections.

The catch is that it does not generalize. A chunker that understands runbooks knows
nothing about Slack threads or vendor PDFs. A real indexing pipeline routes documents
to the right chunker by format, and needs an answer for the format it does not
recognize.


## The dial you should tune

Sentence-window has a `window` parameter, and it is not a cosmetic setting. It is the
precision-versus-context dial, and you can watch it move.


In [ ]:
from rag.pipeline import Chunk

for window in (0, 1, 3):
    chunks = chunk_sentence_window(doc, window=window)
    avg = sum(len(c) for c in chunks) / len(chunks)
    idx = VectorIndex()
    idx.chunks = [Chunk("rb_account_takeover", c, embed(c)) for c in chunks]
    score = idx.retrieve("check for data egress after the compromise", k=1)[0][0]
    print(f"window={window}  {len(chunks)} chunks, avg {avg:3.0f} chars, top score {score:.3f}")


A wider window means each chunk carries more surrounding context — better for the
model's answer, worse for precise matching, because the target sentence is now diluted
by its neighbours.

There is no universally correct window. There is a correct window *for your corpus*,
and the only way to find it is to measure — which is where Part 2 ends.


---
# Part 2 — Advanced RAG

Part 1 is the pipeline everyone builds first. Part 2 is what you add when it meets
real queries, real corpora, and real identifiers — and each addition fixes a failure
you can see.

## Query transformation

Users do not write queries the way documents are written. "Someone hacked us" and
"Account Takeover Response" share no words. Rewriting the user's vocabulary into the
document's vocabulary is often a bigger win than any chunking change.

Two techniques. **Rewriting** maps the query into corpus vocabulary. **Multi-query**
generates several phrasings, retrieves with each, and unions the results — so one
unlucky phrasing cannot sink the search. A real system asks a model for both; the
stand-in here is a lookup table, so the effect is visible and repeatable.


In [ ]:
from rag.hybrid import rewrite_query, multi_query

def top_hit(q):
    hits = index.retrieve(q, k=1)
    return f"{hits[0][1].doc_id:22} {hits[0][0]:.3f}" if hits else "(nothing retrieved - no shared words)"

for query in ["help my account got taken over", "someone hacked us"]:
    rewritten = rewrite_query(query)
    print(f"before: {query!r:38} -> {top_hit(query)}")
    print(f"after:  {rewritten!r:38} -> {top_hit(rewritten)}")
    print()

print("multi-query for 'someone hacked us':")
found = {}
for variant in multi_query("someone hacked us"):
    for score, chunk in index.retrieve(variant, k=1):
        found[chunk.doc_id] = max(found.get(chunk.doc_id, 0), score)
    print(f"  {variant!r}")
print("  union of top hits:", found)


## Hybrid retrieval: where dense search fails

Dense embeddings capture **meaning** and are bad at exact tokens — identifiers, error
codes, CVE numbers. A CVE id is exactly what an analyst searches for and exactly what
an embedding blurs.

The two advisories in the corpus are formulaic and near-identical apart from the
identifier. That is how CVE advisories actually read. The sparse side uses
`rank_bm25`, the standard open-source implementation.

One honest note: the book's toy `embed` is bag-of-words, so it would match the literal
token `2026-2000` and *hide* this failure. `make_semantic_embed` collapses identifiers
the way a real embedding model does in effect — otherwise this section would assert a
failure the code quietly disproves.


In [ ]:
from rag.hybrid import HybridRetriever, make_semantic_embed

# Two formulaic advisories, identical apart from the identifier - which is how
# CVE feeds actually read. Added to the corpus for this section only.
corpus = dict(docs)
corpus["cve_2026_3000"] = ("CVE-2026-3000 advisory. Severity high. Affected component: VPN appliance. "
                           "Mitigation: apply the vendor patch and rotate credentials.")
corpus["cve_2026_4000"] = ("CVE-2026-4000 advisory. Severity high. Affected component: VPN appliance. "
                           "Mitigation: apply the vendor patch and rotate credentials.")

retriever = HybridRetriever(corpus, make_semantic_embed(embed), cosine)
query = "CVE-2026-4000 mitigation"
print(f"query: {query!r}   (the analyst wants ONE specific advisory)")
print()

dense = sorted(retriever.dense(query).items(), key=lambda kv: -kv[1])[:2]
print("dense only  ->", [(d, round(s, 3)) for d, s in dense])
print(f"   dense cannot tell the advisories apart: {abs(dense[0][1] - dense[1][1]) < 0.001}")

sparse = sorted(retriever.sparse(query).items(), key=lambda kv: -kv[1])[:2]
print("sparse only ->", [(d, round(s, 3)) for d, s in sparse])

fused = retriever.search(query, alpha=0.5)[:2]
print("hybrid      ->", [(r["doc"], r["fused"]) for r in fused])
print()
print("Dense scored the two advisories identically - and put the WRONG one first.")
print("BM25 separated them. Neither is better: they fail differently, which is the")
print("argument for running both.")


### The alpha dial, and the fusion that has no dial

`alpha` weights dense against sparse. It is another measured dial that silently sets
quality, and it ships as a vendor default.

**Reciprocal rank fusion** (RRF) is the alternative: it ignores scores and combines
*ranks*, so there is nothing to tune and no normalisation to get wrong. It is what most
production hybrid retrievers actually use.


In [ ]:
print(f'{"alpha":>6}  top result')
for alpha in (0.0, 0.25, 0.5, 0.75, 1.0):
    top = retriever.search(query, alpha=alpha)[0]
    kind = "pure sparse" if alpha == 0 else "pure dense" if alpha == 1 else "hybrid"
    print(f'{alpha:>6}  {top["doc"]:16} ({kind})')

print()
print("reciprocal rank fusion, no weights:")
for r in retriever.rrf(query, k=3):
    print(f'  {r["doc"]:16} {r["rrf"]}')


## Reranking

First-stage retrieval is cheap and approximate: it scores the query and every passage
*separately* and compares vectors. A **reranker** is expensive and precise: it reads
the query and one passage *together* and scores the pair. You cannot afford it on
forty thousand chunks, so you run it on the top ten.

The pattern is retrieve-wide, then rerank-narrow. The stand-in here scores phrase
overlap — it rewards a passage that says what the query says, in the order the query
says it. A real cross-encoder does the same thing with a model.

On five documents the first stage usually has the right passage on top already, so
do not expect the *order* to change here. Watch the **margin**. The first stage
separates the right passage from the runner-up by a fraction; the reranker, reading
query and passage together, separates them by most of the scale. At forty thousand
chunks — where the first stage's top ten hold three near-duplicates and the right
passage at position seven — that margin is what pulls it to position one.


In [ ]:
from rag.advanced import rerank, rerank_score

query = "force a password reset and require re-enrollment of MFA"
candidates = index.retrieve(query, k=6)             # stage 1: wide and cheap

print("first-stage order (cosine):")
for score, chunk in candidates[:4]:
    print(f"  [{score:.3f}] {chunk.doc_id:22} {chunk.text[:60]}...")

print()
print("after reranking (query + passage read together):")
reranked = rerank(query, candidates, top_n=4)             # stage 2: narrow and precise
for score, chunk in reranked:
    print(f"  [{score:.3f}] {chunk.doc_id:22} {chunk.text[:60]}...")

print()
print(f"margin over the runner-up:  first stage {candidates[0][0] - candidates[1][0]:.3f}"
      f"   reranked {reranked[0][0] - reranked[1][0]:.3f}")


## Parent-document retrieval

The chunk-size dilemma from Part 1 has a structural answer. Index *small* chunks (one
sentence) so matching is sharp, but when one matches, hand the model its *parent* —
the whole section — so it has the context to answer.

This is the hierarchical strategy made explicit, and it is what most production RAG
systems converge on for structured documents.


In [ ]:
from rag.advanced import ParentChildIndex

pc = ParentChildIndex(docs, parent_sents=3)
print(f"{len(pc.children)} child sentences indexed, {len(pc.parents)} parent sections")
print()

for score, child, parent in pc.retrieve("revoke active sessions", k=2):
    print(f"matched child  [{score:.3f}] {child.text}")
    print(f"returned parent         {parent[:110]}...")
    print()


## A real vector database, with metadata

`VectorIndex` is a list and a loop. It is enough to learn on and useless at scale.
This section uses **Chroma** — a real vector database from the book's
`requirements.txt` — and shows the three things it adds:

1. **Persistence.** Index once, reopen later, from another process.
2. **Metadata filtering.** "Only advisories." "Only high severity." Filtering *before*
   similarity search is how you stop a phishing runbook from answering a CVE question.
3. **Scale** — approximate nearest-neighbour search (HNSW) instead of a linear scan.

One honesty note: Chroma's default embedding function downloads a model on first use.
To stay offline and deterministic the lab passes its own vectors in explicitly. The
*database* is real; the embedder is still the book's stand-in.


In [ ]:
from rag import vectorstore

STORE = "/tmp/aegis_chroma"
col = vectorstore.build_collection(STORE, docs, METADATA, strategy="semantic")
print(f"collection '{col.name}' persisted to {STORE}: {col.count()} chunks")

reopened = vectorstore.open_collection(STORE)          # a new client, from disk
print(f"reopened from disk: {reopened.count()} chunks")
print()

q = "revoke sessions and rotate tokens"
print(f"query: {q!r}")
print()
print("no filter:")
for doc_id, sim, passage in vectorstore.query(reopened, q, k=2):
    print(f"  [{sim:.3f}] {doc_id:22} {passage[:55]}...")

print()
print("where type == runbook:")
for doc_id, sim, passage in vectorstore.query(reopened, q, k=2, where={"type": "runbook"}):
    print(f"  [{sim:.3f}] {doc_id:22} {passage[:55]}...")

print()
print("where type == advisory:")
for doc_id, sim, passage in vectorstore.query(reopened, q, k=2, where={"type": "advisory"}):
    print(f"  [{sim:.3f}] {doc_id:22} {passage[:55]}...")

print()
print("where severity == critical:")
for doc_id, sim, passage in vectorstore.query(reopened, q, k=2, where={"severity": "critical"}):
    print(f"  [{sim:.3f}] {doc_id:22} {passage[:55]}...")


Metadata is the part of RAG that teams forget until the first wrong answer that was
*retrieved correctly* — the right passage from the wrong document type. Filtering by
metadata is cheaper than any reranker and fixes a class of errors no embedder can.


## Measuring retrieval

Part 1 ended with "the spread is the signal." This is what you do with the signal.

A **golden set** is a list of questions with the document each one should retrieve.
Two metrics: **hit@k** (was the right document in the top *k*?) and **MRR** (mean
reciprocal rank — 1 if it was first, ½ if second, and so on). Run them across the
strategies and the choice stops being an opinion.

The golden set is deliberately mixed: precise queries, vague ones, and one shaped like
an identifier. Expect the hit-rate columns to tie — that is Part 1's lesson again, and
it is why the table also reports the **margin**: the average gap between the right
document's best chunk and the best chunk of any other document. The strategies that
tie on pass/fail separate on margin.


In [ ]:
from rag.advanced import hit_rate, mrr, mean_margin, index_retriever

print(f"golden set: {len(GOLDEN_SET)} queries")
print()
print(f'{"strategy":18} {"hit@1":>6} {"hit@3":>6} {"MRR":>6} {"margin":>7}')
for name in STRATEGIES:
    idx = VectorIndex().index(docs, name)
    r = index_retriever(idx)
    print(f'{name:18} {hit_rate(r, GOLDEN_SET, k=1):6.3f} {hit_rate(r, GOLDEN_SET, k=3):6.3f} '
          f'{mrr(r, GOLDEN_SET):6.3f} {mean_margin(idx, GOLDEN_SET):7.3f}')

print()
print("the same golden set, through query rewriting + hybrid RRF:")
hybrid_docs = HybridRetriever(docs, make_semantic_embed(embed), cosine)
def hybrid_retriever(query, k):
    return [r["doc"] for r in hybrid_docs.rrf(rewrite_query(query), k=k)]
print(f'{"rewrite + hybrid":18} {hit_rate(hybrid_retriever, GOLDEN_SET, k=1):6.3f} '
      f'{hit_rate(hybrid_retriever, GOLDEN_SET, k=3):6.3f} {mrr(hybrid_retriever, GOLDEN_SET):6.3f}')
print()
print("which queries the plain semantic index gets wrong at k=1:")
plain = index_retriever(VectorIndex().index(docs, "semantic"))
for query, expected in GOLDEN_SET:
    got = (plain(query, 1) or ["(nothing retrieved)"])[0]
    if got != expected:
        print(f"  {query!r:52} got {got}, wanted {expected}")


One query defeats every embedding strategy: "someone hacked us" shares no words with
any document, so a vocabulary-matching retriever returns nothing at all. Rewriting
closes that gap, which is why the hybrid row is perfect. That is the shape of most
real retrieval failures — not a wrong ranking, but a query the corpus does not speak.

Eight queries is a toy golden set. Fifty is a real one, and building it — from the
questions analysts actually ask — is the highest-leverage afternoon in any RAG
project. Chapter 10 does the full evaluation treatment, including the LLM-judged
metrics (faithfulness, answer relevancy) that the deterministic ones here cannot
cover.


## Optional — real RAGAS

Everything above is retrieval. Measuring whether the *answers* are grounded is
evaluation, and RAGAS is the standard open-source library for it. It is already
installed from `requirements.txt` — and the reason `check_env` guards the
`langchain-community` pin is that `ragas` imports a module that newer versions
removed. A green install followed by a broken import is the failure the pin prevents.

Two of its metrics are deterministic — real library, no model, no key, no cost. They
score the answer against a reference, which is a different question from grounding,
and a useful one.


In [ ]:
try:
    from ragas.metrics.collections import BleuScore, ExactMatch
except Exception as exc:           # keep the lab runnable if ragas is unavailable
    print(f"ragas not importable here ({type(exc).__name__}); skipping. See check_env's pin note.")
else:
    bleu, exact = BleuScore(), ExactMatch()
    REFERENCE = "Disable the affected account and revoke all active sessions."
    for label, response in (("grounded  ", REFERENCE),
                            ("partial   ", "Disable the account and revoke sessions."),
                            ("fabricated", "Deploy the zero-trust mesh and rotate the HSMs.")):
        b = bleu.score(reference=REFERENCE, response=response)
        e = exact.score(reference=REFERENCE, response=response)
        print(f"{label}  BleuScore={b.value:.3f}   ExactMatch={e.value:.0f}")


---

## What you built

A complete RAG pipeline — chunk, embed, retrieve, ground — and then everything a
production system layers on top of it: query rewriting, hybrid dense + sparse
retrieval with two fusion methods, a reranker, parent-document retrieval, a real
vector database with metadata filters, a grounding check with citations, and the
metrics that make the choices measurable.

Six things to carry forward:

- **Chunking is the biggest lever in RAG,** and it is the stage teams skip past on
  their way to arguing about embedding models.
- **On a small corpus, everything works.** The pass/fail column is nearly useless; the
  confidence *margin* predicts behaviour at scale, and a golden set turns it into a
  number.
- **Dense and sparse fail differently.** That is why you run both — and why the
  identifier your analyst is searching for needs BM25.
- **Retrieve wide, rerank narrow.** Precision you cannot afford everywhere you can
  afford on the top ten.
- **Metadata fixes what embeddings cannot.** The right passage from the wrong document
  type is still the wrong answer.
- **Retrieval failure is silent.** No exception, no alert: just a confident answer
  grounded in the wrong document. The grounding check is how you hear it.

**Next:** Chapter 7 gives Aegis a plan — it decides on a multi-step investigation up
front, replans when a log source goes down, and reflects on whether its evidence
actually supports the verdict it reached.
